In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

In [ ]:
student_df = pd.read_csv('/content/gdrive/MyDrive/mental_health_prdiction/cleaned_student_dataset.csv')
reddit_df = pd.read_csv('/content/gdrive/MyDrive/mental_health_prdiction/cleaned_reddit_dataset.csv')

In [ ]:
student_df.head()

In [ ]:
student_df = student_df.drop(['serialized_text'], axis=1)

In [ ]:
student_df.head()

In [ ]:
student_df.shape

In [ ]:
def serialize_student_row(row):
    """
    Converts your specific columns into a clinical text sentence.
    """
    # Handling floats by converting to int first
    pressure_map = {0: 'no', 1: 'very low', 2: 'low', 3: 'moderate', 4: 'high', 5: 'extreme'}

    # int(float(...)) to handle '5.0' appearing as a string or float
    try:
        academic_pressure = pressure_map.get(int(float(row.get('Academic Pressure', 3))), 'moderate')
        work_pressure = pressure_map.get(int(float(row.get('Work Pressure', 0))), 'low')
        financial_stress  = pressure_map.get(int(float(row.get('Financial Stress', 0))), 'low')
        family_history = 'Yes' if int(float(row.get('Family History of Mental Illness', 0))) == 1 else 'No'
    except:

        academic_pressure, work_pressure, financial_stress, family_history = 'moderate', 'low', 'low', 'No'

    # sentence Build
    text = (
        f"[CLINICAL] Subject is a {int(float(row.get('Age', 20)))}-year-old {row.get('Gender', 'Student')}. "
        f"Academic standing: CGPA of {row.get('CGPA', 0.0)}. "
        f"They report {academic_pressure} academic pressure and {work_pressure} work pressure. "
        f"Sleep duration: {row.get('Sleep Duration', 'unknown')}. "
        f"Dietary habits: {row.get('Dietary Habits', 'unknown')}. "
        f"Financial stress is {financial_stress}. "
        f"Family history of mental illness: {family_history}."
    )
    return text

In [ ]:
# translation
student_df['text'] = student_df.apply(serialize_student_row, axis=1)

In [ ]:
# FUSING DATASETS
# Prepare Student Subset (Text + Label)
student_subset = student_df[['text', 'Depression']].rename(columns={'Depression': 'labels'})
# Prepare Reddit Subset (Text + Label)
reddit_subset = reddit_df[['clean_text', 'binary_label']].rename(columns={'clean_textt': 'text', 'binary_label': 'labels'})

# Vertical Stack
full_dataset = pd.concat([student_subset, reddit_subset], axis=0)

# Shuffle & Split
full_dataset = full_dataset.sample(frac=1, random_state=42).reset_index(drop=True)
train_df, test_df = train_test_split(full_dataset, test_size=0.2, random_state=42, stratify=full_dataset['labels'])

In [ ]:
#Missing Values Instead of Dropping and Fill NaNs with empty strings
reddit_subset['clean_text'] = reddit_subset['clean_text'].fillna("Empty Post")
student_subset['text'] = student_subset['text'].fillna("Empty Note")

In [ ]:
# CLEANING(some Reddit dataset has some empty rows or numbers mixed in with the posts)

# we Convert everything to String
train_df['text'] = train_df['text'].astype(str)
test_df['text'] = test_df['text'].astype(str)

# we Remove any rows that are still empty or just "nan"
train_df = train_df[train_df['text'].str.strip() != 'nan']
test_df = test_df[test_df['text'].str.strip() != 'nan']
train_df = train_df[train_df['text'].notna()]
test_df = test_df[test_df['text'].notna()]

In [ ]:
from huggingface_hub import login
login()

In [ ]:
#TOKENIZATION
model_id = "mental/mental-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

hf_train = Dataset.from_pandas(train_df)
hf_test = Dataset.from_pandas(test_df)

tokenized_train = hf_train.map(tokenize_function, batched=True)
tokenized_test = hf_test.map(tokenize_function, batched=True)

In [ ]:
# TRAINING
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None): 
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Weight class 1 (Depressed) higher to ensure we catch it
        weights = torch.tensor([1.0, 2.0]).to(model.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

training_args = TrainingArguments(
    output_dir="/content/gdrive/MyDrive/mental_health_prediction/unified_results",
    num_train_epochs=3,            
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    report_to="none"
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)
trainer.train()

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# Predictions on the Test Set
predictions = trainer.predict(tokenized_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Calculate Metrics
accuracy = accuracy_score(labels, preds)
recall = recall_score(labels, preds)
precision = precision_score(labels, preds)
f1 = f1_score(labels, preds)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Save the Model
save_path = "/content/gdrive/MyDrive/mental_health_prediction/final_unified_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model successfully saved to: {save_path}")
